# Path B — A+B notebook (10 epochs each, no oracle)

Runs the two control arms of the Path B ablation. Pair with `train_path_b_colab_CD.ipynb` (running in parallel on a second Colab) to cover all four runs in ~24h.

| run | --oracle-lambda | --color-augment | purpose |
|---|---|---|---|
| **A** | 0.0  | no  | V12-only baseline (closest reproducer of pillar2z; isolates leakage-fix delta) |
| **B** | 0.0  | yes | V12 + color aug (isolates color-symmetry gain on top of A) |

**Batch size:** 32768 (same as pillar2z). If you OOM, drop to 16384 + halve `--lr`. If on A100-80GB with headroom, you can try 49152 — but the val_loss won't be directly comparable to pillar2z any more.

**Wall time estimate per run:**
- A100-80GB: ~5h
- A100-40GB: ~6h
- L4 (22GB): ~7-8h

## Upload to Drive (`MyDrive/alphatrain/`)

1. `colorlines_path_b.tar.gz` — code + oracle tensor (~1.1 MB)
2. `v12_pillar2z.pt.gz` — already there from pillar2z run (~2-3 GB)
3. `pillar2y2_epoch_40.pt` — already there (~143 MB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar2y2_epoch_40.pt',
            '/content/alphatrain/data/model.pt')
print(f'Warm-start: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

t0 = time.time()
!gzip -dc {DRIVE}/v12_pillar2z.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'V12 tensor: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

%cd /content
!python -m pytest alphatrain/tests/test_train_path_b.py -v --tb=short 2>&1 | tail -20

## Run A — V12-only baseline, no color aug, no oracle

Closest reproducer of pillar2z. Isolates any difference attributable to the new pipeline (`train_path_b.py` λ=0 path vs original `train.py`). If A's epoch-10 val_loss does NOT match pillar2z's epoch-10 val_loss (within fp16 noise), the new code path has a bug.

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --epochs 10 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --oracle-lambda 0.0 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_A_smoke_best.pt \
    --save-dir /content/checkpoints/path_b_A_smoke

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_A_smoke'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')

## Run B — V12 + color aug, no oracle

Adds color-permutation augmentation on top of A. If B > A at epoch 10, color symmetry is paying off. If B ≈ A, color aug isn't moving the needle at this corpus size.

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 10 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --oracle-lambda 0.0 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_B_smoke_best.pt \
    --save-dir /content/checkpoints/path_b_B_smoke

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_B_smoke'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')